# Chapter 1: CCTV Recording and Storage Fundamentals

**Companion notebook for Intelligent Video Analytics, Chapter 1 (pp. 1 to 21).**  
**www.intelligentxhub.com**

This notebook demonstrates CCTV storage estimation, recording modes, capacity planning and simple event-metadata retrieval. No live cameras, personal data or API keys are used.

**Learning outcomes**

1. Calculate the effect of bit rate, camera count and retention.
2. Compare continuous, scheduled and event-triggered recording assumptions.
3. Distinguish decimal GB, binary GiB and the book's baseline expression.
4. Create and query a small synthetic event log.

Run all cells from top to bottom. Change the parameters to complete the questions at the end.

## 1. Storage formulas

Chapter 1, page 16 gives a simplified equation for storage based on bit rate and retention. We distinguish three calculations:

- **Book-style expression**: `Mbps * 3600 * hours/day * days / (8 * 1024)`.
- **Decimal GB**: `Mbps * 1,000,000 * seconds / (8 * 1,000,000,000)`.
- **Binary GiB**: divide stored bytes by `1024 ** 3`.

The book-style expression is retained for continuity, but it mixes decimal Mbps with a binary 1024 divisor. Use GB and GiB calculations with explicit units when sizing equipment. All three exclude data overhead and redundancy.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
import math
import pandas as pd
import matplotlib.pyplot as plt

SECONDS_PER_HOUR = 3600
HOURS_PER_DAY = 24
BITS_PER_BYTE = 8
MBPS_TO_BPS = 1_000_000
BYTES_PER_GB = 1_000_000_000
BYTES_PER_GIB = 1024 ** 3

@dataclass(frozen=True)
class CCTVScenario:
    cameras: int = 20
    bitrate_mbps: float = 2.0
    retention_days: int = 30
    recorded_hours_per_day: float = 24.0
    active_fraction: float = 1.0
    usable_fraction: float = 0.8

    def __post_init__(self):
        assert self.cameras > 0
        assert self.bitrate_mbps > 0
        assert self.retention_days > 0
        assert 0 < self.recorded_hours_per_day <= 24
        assert 0 <= self.active_fraction <= 1
        assert 0 < self.usable_fraction <= 1

    @property
    def seconds_recorded(self):
        return self.recorded_hours_per_day * SECONDS_PER_HOUR * self.retention_days * self.active_fraction

    def payload_bytes(self):
        return self.cameras * self.bitrate_mbps * MBPS_TO_BPS * self.seconds_recorded / BITS_PER_BYTE

    def decimal_gb(self):
        return self.payload_bytes() / BYTES_PER_GB

    def binary_gib(self):
        return self.payload_bytes() / BYTES_PER_GIB

    def book_style_gb(self):
        return self.cameras * self.bitrate_mbps * SECONDS_PER_HOUR * self.recorded_hours_per_day * self.retention_days * self.active_fraction / (8 * 1024)

    def raw_required_gb(self):
        return self.decimal_gb() / self.usable_fraction

s = CCTVScenario()
print('Scenario:', s)
print(f'Book-style estimate: {s.book_style_gb():,.2f} (book-style GB)')
print(f'Exact decimal payload: {s.decimal_gb():,.2f} GB')
print(f'Exact binary payload: {s.binary_gib():,.2f} GiB')
print(f'Raw capacity at {s.usable_fraction:.0%} usable: {s.raw_required_gb():,.2f} GB')

### Verification against the chapter example

Page 16 uses 2 Mbps, 30 days and 20 cameras. Direct substitution into the book-style expression yields **632.8125 per camera**, or **12,656.25 for 20 cameras**, not the printed 633.79 per camera. For storage procurement, prefer explicit unit conversions and vendor-specific measurements.

In [ ]:
single = CCTVScenario(cameras=1, bitrate_mbps=2, retention_days=30)
assert abs(single.book_style_gb() - 632.8125) < 1e-9
assert abs(single.decimal_gb() - 648.0) < 1e-9
assert abs(single.binary_gib() - (648_000_000_000 / 1024**3)) < 1e-9
assert abs(CCTVScenario(cameras=20).book_style_gb() - 12_656.25) < 1e-8
print('Storage formula checks passed.')

## 2. Compare three recording strategies

Chapter 1 (pp. 14 to 15) discusses continuous, scheduled and event-triggered recording. The duty cycle is an assumed share of time when data is stored. It is not a measured detection rate.

Modify the duration or active fraction and recalculate.

In [ ]:
strategies = {
    'Continuous': CCTVScenario(recorded_hours_per_day=24, active_fraction=1.0),
    'Scheduled (12 h/day)': CCTVScenario(recorded_hours_per_day=12, active_fraction=1.0),
    'Event-triggered (30% duty)': CCTVScenario(recorded_hours_per_day=24, active_fraction=0.30),
}
rows = []
for name, cfg in strategies.items():
    rows.append({'Mode': name, 'Payload GB': cfg.decimal_gb(), 'Raw capacity GB': cfg.raw_required_gb()})
strategy_df = pd.DataFrame(rows)
display(strategy_df.round(2))
ax = strategy_df.plot(kind='bar', x='Mode', y='Payload GB', legend=False, rot=0, figsize=(9, 4))
ax.set_ylabel('Decimal GB')
ax.set_title('Estimated storage by recording mode')
plt.tight_layout()
plt.show()

## 3. Storage planning across a camera inventory

Different surveillance locations need different video bit rates, recording windows and retention periods. Build a mixed camera inventory instead of applying one setting to every camera.

In [ ]:
inventory = pd.DataFrame([
    {'location':'Entrance','cameras':4,'bitrate_mbps':4,'hours_per_day':24,'active_fraction':1.0,'days':30},
    {'location':'Parking','cameras':6,'bitrate_mbps':2,'hours_per_day':24,'active_fraction':0.6,'days':30},
    {'location':'Warehouse','cameras':8,'bitrate_mbps':2,'hours_per_day':12,'active_fraction':1.0,'days':14},
    {'location':'Lobby','cameras':2,'bitrate_mbps':3,'hours_per_day':24,'active_fraction':1.0,'days':30},
])
def estimate_gb(row):
    cfg = CCTVScenario(cameras=int(row['cameras']), bitrate_mbps=float(row['bitrate_mbps']),
        retention_days=int(row['days']), recorded_hours_per_day=float(row['hours_per_day']),
        active_fraction=float(row['active_fraction']))
    return cfg.decimal_gb()
inventory['payload_GB'] = inventory.apply(estimate_gb, axis=1)
display(inventory.round(2))
print(f'Total payload: {inventory.payload_GB.sum():,.2f} GB')
print(f'Raw storage with 80% usable allocation: {inventory.payload_GB.sum()/0.8:,.2f} GB')

## 4. Sensitivity analysis

Change bit rate and retention to see their linear effect on theoretical storage. This is a useful first check, not a codec performance benchmark.

In [ ]:
bit_rates = [1, 2, 4, 6, 8]
retentions = [7, 14, 30, 60, 90]
sensitivity = pd.DataFrame(index=[f'{b} Mbps' for b in bit_rates], columns=[f'{d} d' for d in retentions], dtype=float)
for b in bit_rates:
    for d in retentions:
        sensitivity.loc[f'{b} Mbps', f'{d} d'] = CCTVScenario(cameras=10, bitrate_mbps=b, retention_days=d).decimal_gb()
display(sensitivity.round(1))
plt.figure(figsize=(9,4))
for b in bit_rates:
    plt.plot(retentions, sensitivity.loc[f'{b} Mbps'].values.astype(float), marker='o', label=f'{b} Mbps')
plt.xlabel('Retention (days)')
plt.ylabel('Payload (GB) for 10 cameras')
plt.title('Storage sensitivity')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Synthetic metadata indexing and retrieval

Chapter 1 (pp. 18 to 19) describes timestamps, camera IDs, locations and event tags. This example uses a deterministic, fictitious dataset. It does not perform facial recognition or operate a live surveillance system.

In [ ]:
events = pd.DataFrame([
    ('2026-01-02 08:15:00', 'CAM-01','Entrance','person_entered'),
    ('2026-01-02 08:18:00', 'CAM-03','Parking','vehicle_parked'),
    ('2026-01-02 09:04:00', 'CAM-01','Entrance','motion_detected'),
    ('2026-01-02 09:08:00', 'CAM-04','Warehouse','motion_detected'),
    ('2026-01-02 09:12:00', 'CAM-03','Parking','vehicle_departed'),
    ('2026-01-02 10:10:00', 'CAM-01','Entrance','person_entered'),
], columns=['timestamp','camera_id','location','event_type'])
events['timestamp'] = pd.to_datetime(events['timestamp'])
result = events.loc[(events['camera_id']=='CAM-01') & (events['timestamp'].dt.hour.between(8, 9))]
display(events)
print('Filtered entrance events between 08:00 and 09:59:')
display(result)
assert len(result)==2

## 6. Student questions

1. Repeat the default 20-camera scenario at 4 Mbps. What changes and why?
2. For a 16-camera installation, compare 7, 30 and 90 days of retention.
3. If the operator records only 10 hours per day, what assumptions must be checked before using the result for procurement?
4. Explain why event-triggered recording may miss incidents. How could pre-event recording or buffer settings help?
5. Modify the inventory so that parking cameras retain video for 60 days. Report the revised decimal GB total.
6. Add a camera to the event log and retrieve only its events.
7. Explain the distinction between recorded video footage and metadata that indexes it.
8. Why is it incorrect to treat the theoretical payload as a complete RAID or cloud-storage procurement specification?

**Submission:** Provide your scenario assumptions, table of calculations, two plots and a concise technical conclusion.

**Source:** Mahendra, Adinandra and Wisudawan, *Intelligent Video Analytics*, Chapter 1, pp. 14 to 19.  
**www.intelligentxhub.com**